In [56]:
import pandas as pd


In [57]:

with open('../src/.output/graph_edges_tfl.csv', mode='r') as file:
    edges = pd.read_csv(file)

stations = pd.read_parquet('../src/.output/timetable_tfl/stations.parquet')


In [58]:
edges = edges.merge(stations, left_on="source", right_on="name", how="left" )
edges = edges.merge(stations, left_on="target", right_on="name", how="left", suffixes=("_source", "_target"))

edges = edges.drop(columns=['name_source', 'name_target'])
edges = edges.dropna(how="any",axis=0)

In [59]:
edges.head(5)

,source,target,value,id_source,lat_source,lon_source,id_target,lat_target,lon_target
0,Harrow & Wealdstone Underground Station,Kenton Underground Station,1.722210,940GZZLUHAW,51.592268,-0.335217,940GZZLUKEN,51.581756,-0.316910
1,Kenton Underground Station,South Kenton Underground Station,1.408965,940GZZLUKEN,51.581756,-0.316910,940GZZLUSKT,51.570232,-0.308433
2,South Kenton Underground Station,North Wembley Underground Station,0.907389,940GZZLUSKT,51.570232,-0.308433,940GZZLUNWY,51.562551,-0.304000
3,North Wembley Underground Station,Wembley Central Underground Station,1.241961,940GZZLUNWY,51.562551,-0.304000,940GZZLUWYC,51.552304,-0.296852
4,Wembley Central Underground Station,Stonebridge Park Underground Station,1.720930,940GZZLUWYC,51.552304,-0.296852,940GZZLUSGP,51.543959,-0.275892


In [60]:
# Combine both columns into one Series
all_nodes = pd.concat([edges["source"], edges["target"]])

# Drop duplicates
unique_nodes = all_nodes.drop_duplicates().reset_index(drop=True)

print(len(unique_nodes))

311


In [61]:
from pyvis.network import Network

net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white")
net.barnes_hut()

# Add nodes
for node in unique_nodes:
    net.add_node(node, label=node, title=node)

# Add edges
for _, edge in edges.iterrows():
    net.add_edge(edge["source"], edge["target"], value=edge.get("distance", 1))

# Build neighbor map
neighbor_map = net.get_adj_list()

# Add hover info
for node in net.nodes:
    neighbors = neighbor_map.get(node["id"], [])
    node["title"] += " Neighbors:<br>" + "<br>".join(neighbors)
    node["value"] = len(neighbors)

# Show
net.show("network.html", notebook=False)


network.html


## Graph of the Network

In [62]:
import plotly.graph_objects as go

import networkx as nx

G = nx.random_geometric_graph(unique_nodes.__len__(), 0.125)
print(unique_nodes.__len__())

311


In [63]:
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = G.nodes[edge[0]]['pos']
    x1, y1 = G.nodes[edge[1]]['pos']
    edge_x.append(x0)
    edge_x.append(x1)
    edge_x.append(None)
    edge_y.append(y0)
    edge_y.append(y1)
    edge_y.append(None)

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

node_x = []
node_y = []
for node in G.nodes():
    x, y = G.nodes[node]['pos']
    node_x.append(x)
    node_y.append(y)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    hoverinfo='text',
    marker=dict(
        showscale=True,
        # colorscale options
        #'Greys' | 'YlGnBu' | 'Greens' | 'YlOrRd' | 'Bluered' | 'RdBu' |
        #'Reds' | 'Blues' | 'Picnic' | 'Rainbow' | 'Portland' | 'Jet' |
        #'Hot' | 'Blackbody' | 'Earth' | 'Electric' | 'Viridis' |
        colorscale='YlGnBu',
        reversescale=True,
        color=[],
        size=10,
        colorbar=dict(
            thickness=15,
            title=dict(
              text='Node Connections',
              side='right'
            ),
            xanchor='left',
        ),
        line_width=2))

In [64]:
node_adjacencies = []
node_text = []
for node, adjacencies in enumerate(G.adjacency()):
    node_adjacencies.append(len(adjacencies[1]))
    node_text.append('# of connections: '+str(len(adjacencies[1])))

node_trace.marker.color = node_adjacencies
node_trace.text = node_text

In [65]:
import plotly.graph_objects as go

fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title=dict(
                    text="<br>Network graph made with Python",
                    font=dict(
                        size=16
                    )
                ),
                showlegend=False,
                hovermode='closest',
                margin=dict(b=20,l=5,r=5,t=40),
                annotations=[ dict(
                    text="Python code: <a href='https://plotly.com/python/network-graphs/'> https://plotly.com/python/network-graphs/</a>",
                    showarrow=False,
                    xref="paper", yref="paper",
                    x=0.005, y=-0.002 ) ],
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                )
fig.show()